# From Raw Events to Gold Insights in Minutes

**A 10-cell end-to-end demo of the medallion pipeline.**

This notebook walks through the complete medallion architecture in a single, self-contained session:

1. **Raw IoT events** (JSON lines from Cumulocity) → **Silver Iceberg table** (written with PyIceberg)
2. **Silver table** → **Spark transformation** (join with asset hierarchy, hourly aggregation)
3. **Spark output** → **Gold Iceberg table** (aggregated, business-ready, queryable with SQL)

Each step takes one or two cells. By the end you will have a working medallion pipeline on your laptop.

In [ ]:
import sys
import json
import shutil
from pathlib import Path
import pyarrow as pa
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog

sys.path.insert(0, str(Path.cwd()))

# Clean up from previous runs
for d in ["../data/warehouse_silver", "../data/warehouse_gold"]:
    shutil.rmtree(d, ignore_errors=True)
for fn in ["../data/warehouse_silver/silver_catalog.db"]:
    Path(fn).unlink(missing_ok=True)

# Create silver catalog and table
catalog = SqlCatalog("silver", **{
    "uri": "sqlite:///../data/warehouse_silver/silver_catalog.db",
    "warehouse": "file://" + str(Path("../data/warehouse_silver").resolve()),
})
catalog.create_namespace_if_not_exists("silver")

# Read events.jsonl
events = []
with open("../data/input/events.jsonl") as f:
    for line in f:
        events.append(json.loads(line))

df_events = pd.DataFrame(events)
arrow_table = pa.Table.from_pandas(df_events)

# Create silver events table
silver_events = catalog.create_table("silver.events", schema=arrow_table.schema)
silver_events.append(arrow_table)

snapshot = silver_events.current_snapshot()
print(f"Silver table created: {len(events):,} events")
print(f"Snapshot ID: {snapshot.snapshot_id}")
print(f"Table location: {silver_events.location()}")

## Start Spark and read the silver table

In [ ]:
from helpers import get_spark

spark = get_spark("FlashDemo", gold_warehouse="../data/warehouse_gold")

# Convert PyIceberg silver table to a Spark DataFrame via Apache Arrow (zero-copy).
# PyIceberg's SqlCatalog uses SQLite for metadata — not compatible with Spark's
# Hadoop catalog convention. Arrow is the common in-memory format between the two.
silver_df = spark.createDataFrame(silver_events.scan().to_arrow().to_pandas())

print(f"Silver rows: {silver_df.count():,}")
silver_df.show(5, truncate=True)

## Build a device→group lookup from the asset hierarchy

In [ ]:
import json

# Load device metadata
devices = []
with open("../data/input/cmdata.jsonl") as f:
    for line in f:
        devices.append(json.loads(line))

# Flatten: for each parent, assign its childAssets as children
hierarchy = []
for d in devices:
    parent_id = str(d["id"])
    parent_name = d.get("name", parent_id)
    for child_id in d.get("childAssets", []) + d.get("childDevices", []):
        hierarchy.append({"device_id": str(child_id), "group_id": parent_id, "group_name": parent_name})

# Create Spark DataFrame
hierarchy_df = spark.createDataFrame(hierarchy)
hierarchy_df = hierarchy_df.dropDuplicates(["device_id"])
print(f"Hierarchy entries: {hierarchy_df.count():,}")
hierarchy_df.show(5)

## Aggregate: hourly event counts per device group

In [ ]:
from pyspark.sql import functions as F

# Join events with hierarchy
enriched = silver_df.join(
    hierarchy_df,
    silver_df["source"] == hierarchy_df["device_id"],
    "left"
).withColumn(
    "event_hour",
    F.date_trunc("hour", F.to_timestamp("time"))
).withColumn(
    "group_name",
    F.coalesce(F.col("group_name"), F.lit("ungrouped"))
)

# Gold: hourly event counts per group
gold_df = enriched.groupBy("event_hour", "group_id", "group_name").agg(
    F.count("*").alias("event_count"),
    F.countDistinct("source").alias("active_devices")
).orderBy("event_hour", "group_name")

print(f"Gold rows: {gold_df.count():,}")
gold_df.show(10, truncate=False)

In [ ]:
# Write to Iceberg gold table via Spark Hadoop catalog
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")
gold_df.writeTo("local.gold.device_hourly").createOrReplace()

# Query with Spark SQL
spark.sql("""
    SELECT group_name,
           COUNT(DISTINCT event_hour) AS hours_active,
           SUM(event_count)           AS total_events,
           MAX(active_devices)        AS peak_devices
    FROM   local.gold.device_hourly
    GROUP  BY group_name
    ORDER  BY total_events DESC
    LIMIT  10
""").show(truncate=False)

## What just happened

In ten cells you built a complete medallion pipeline. Here is what each step did:

- **Silver table**: PyIceberg read raw `events.jsonl` and wrote it as an Iceberg table with a full snapshot. Spark read it directly by filesystem path — no shared catalog needed.
- **Spark transformation**: PySpark joined events with the asset hierarchy from `cmdata.jsonl`, truncated timestamps to hourly granularity, and aggregated event counts and active device counts per group per hour.
- **Gold table**: Spark wrote the aggregated result as an Iceberg table via its Hadoop-backed catalog (`local`). It is queryable with standard SQL immediately after write.

This three-step pattern — **ingest → transform → serve** — is the medallion architecture. The rest of this module explores each layer in depth: how silver tables are structured and maintained, how Spark handles large-scale transformations, and how the gold layer maps to classical data warehouse concepts like star schemas and data marts.